Data playground for custom tracks and annotations

In [ ]:
import pandas as pd
import yt_dlp
import os
import re
import time
import random
import shutil
from pathlib import Path

def sanitize_filename(name):
    # Remove invalid filesystem characters
    return re.sub(r'[\\/*?:"<>|]', "", name)

def download_track_from_youtube(row, output_dir="../data/custom/audio"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    artist = str(row['Artist name'])
    track = str(row['Track name'])
    album = str(row['Album']) if not pd.isna(row['Album']) else ""
    
    # Create the filename
    filename_base = sanitize_filename(f"{artist} - {track}")
    output_path = os.path.join(output_dir, f"{filename_base}.mp3")
    
    # Skip if already downloaded
    if os.path.exists(output_path):
        # print(f"Exists: {filename_base}")
        return True

    # Robust FFmpeg finding (borrowed from MDJCUE notebook)
    ffmpeg_location = shutil.which('ffmpeg')
    ffmpeg_parent = str(Path(ffmpeg_location).parent) if ffmpeg_location else None

    # Filter function to skip long videos efficiently
    def check_duration(info, *, incomplete):
        duration = info.get('duration')
        if duration and duration > 600:
            return 'Video too long (> 10 mins)'
        return None

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': os.path.join(output_dir, f"{filename_base}.%(ext)s"),
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'quiet': True,
        'noplaylist': True,
        'default_search': 'ytsearch1',
        'match_filter': check_duration,
        'ignoreerrors': False, # Let us catch the error to try fallback
        'no_warnings': True,
        'nocheckcertificate': True,
        'extractor_args': {
            'youtube': {
                'player_client': ['android'],
                'skip': ['hls', 'dash']
            }
        },
    }
    
    if ffmpeg_parent:
        ydl_opts['ffmpeg_location'] = ffmpeg_parent

    # Queries to try (Specific -> General)
    # Sanitizing queries to remove characters that might be interpreted as URL schemes (like ":")
    safe_artist = artist.replace(":", " ")
    safe_track = track.replace(":", " ")
    safe_album = album.replace(":", " ")
    
    queries = [
        f"{safe_artist} - {safe_track} {safe_album} audio", # First try specific
        f"{safe_artist} - {safe_track} audio",         # Fallback: Just Artist - Track
        f"{safe_artist} - {safe_track}"                # Fallback: Broadest
    ]

    for attempt, query in enumerate(queries):
        try:
            print(f"Processing: {artist} - {track} (Attempt {attempt+1})...")
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.extract_info(query, download=True)
            
            if os.path.exists(output_path):
                time.sleep(random.uniform(2, 5))
                return True
            else:
                print(f"Warning: Download ran but file {filename_base}.mp3 not found. (Likely filtered or format issue)")
                # If we are on the last attempt and still no file, we return False at the end of loop
            
        except Exception as e:
            # If it's the last attempt, print error
            if attempt == len(queries) - 1:
                print(f"Failed all attempts for {artist} - {track}: {e}")
                return False
            # Otherwise continue to next query
            # print(f"Attempt {attempt+1} failed ({e}), retrying simpler query...")
            time.sleep(1)

    return os.path.exists(output_path)

# --- Execution ---
csv_file_path = "../data/custom/house_music_personal.csv"

if os.path.exists(csv_file_path):
    print(f"Reading CSV: {csv_file_path}")
    df = pd.read_csv(csv_file_path)
else:
    print(f"Error: CSV not found at {csv_file_path}")

Reading CSV: ../data/custom/house_music_personal.csv


In [ ]:
import pandas as pd
import os
import re

# Paths
csv_file_path = "../data/custom/house_music_personal.csv"
audio_dir = "../data/custom/audio"

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "", name)

if os.path.exists(csv_file_path):
    print("Running Cross-Reference Check...")
    df = pd.read_csv(csv_file_path)
    
    found_count = 0
    missing_rows = []
    
    for index, row in df.iterrows():
        artist = str(row['Artist name'])
        track = str(row['Track name'])
        
        filename_base = sanitize_filename(f"{artist} - {track}")
        expected_path = os.path.join(audio_dir, f"{filename_base}.mp3")
        
        if os.path.exists(expected_path):
            found_count += 1
        else:
            missing_rows.append(row)
            
    # Summary
    print("-" * 40)
    print(f"Total Tracks in CSV: {len(df)}")
    print(f"Found Audio Files:   {found_count}")
    print(f"Missing Audio Files: {len(missing_rows)}")
    print("-" * 40)
    
    if len(missing_rows) > 0:
        print(f"Attempting to re-download {len(missing_rows)} missing items...")
        
        success_redownload = 0
        for row in missing_rows:
            try:
                # download_track_from_youtube now returns Boolean success
                if download_track_from_youtube(row):
                    success_redownload += 1
            except KeyboardInterrupt:
                print("Re-download stopped by user.")
                break
            except Exception as e:
                print(f"Critical Loop Error for {row['Artist name']}: {e}")
                
        print(f"Re-download session complete. Recovered {success_redownload}/{len(missing_rows)} tracks.")
    else:
        print("All tracks accounted for!")

else:
    print(f"CSV Failure: {csv_file_path} not found.")

Running Cross-Reference Check...
----------------------------------------
Total Tracks in CSV: 127
Found Audio Files:   126
Missing Audio Files: 1
----------------------------------------
Attempting to re-download 1 missing items...
Processing: Rhye - Waste (RY X Remix) (Attempt 1)...
Re-download stopped by user.
Re-download session complete. Recovered 0/1 tracks.


In [1]:
import json
import os
import pandas as pd
import re

# Try to use librosa for duration if available, else 0
try:
    import librosa
    HAS_LIBROSA = True
except ImportError:
    HAS_LIBROSA = False
    print("Librosa not found, duration will be 0.")

# Paths
csv_file_path = "../data/custom/house_music_personal.csv"
audio_dir = "../data/custom/audio"
annotations_dir = "../data/custom/annotations"

# Ensure output dir exists
os.makedirs(annotations_dir, exist_ok=True)

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "", name)

def create_jams_blueprint(row, overwrite=False):
    artist = str(row['Artist name'])
    track = str(row['Track name'])
    
    filename_base = sanitize_filename(f"{artist} - {track}")
    audio_path = os.path.join(audio_dir, f"{filename_base}.mp3")
    jams_path = os.path.join(annotations_dir, f"{filename_base}.jams")
    
    # Skip if JAMS already exists (unless overwrite is True)
    if os.path.exists(jams_path) and not overwrite:
        return "Exists", filename_base

    # Get duration if audio exists
    duration = 0.0
    if HAS_LIBROSA and os.path.exists(audio_path):
        try:
            # Load only duration (fast)
            duration = librosa.get_duration(path=audio_path)
        except Exception as e:
            print(f"Warning: Could not get duration for '{filename_base}': {e}")
            
    # MDJCUE Structure Blueprint with Predefined Points
    blueprint = {
        "annotations": [
            {
                "annotation_metadata": {
                    "curator": { "name": "", "email": "" },
                    "annotator": { "name": "User" },
                    "version": "1.0",
                    "corpus": "Custom House",
                    "annotation_tools": "Manual",
                    "annotation_rules": "",
                    "validation": "",
                    "data_source": "User"
                },
                "namespace": "cue_point",
                "data": [
                    # 3 Entry Points (IN)
                    { "time": 0.0, "duration": 0.0, "value": { "label": "IN", "comment": "In 1" }, "confidence": None },
                    { "time": 0.0, "duration": 0.0, "value": { "label": "IN", "comment": "In 2" }, "confidence": None },
                    { "time": 0.0, "duration": 0.0, "value": { "label": "IN", "comment": "In 3" }, "confidence": None },
                    # 3 Exit Points (OUT)
                    { "time": 0.0, "duration": 0.0, "value": { "label": "OUT", "comment": "Out 1" }, "confidence": None },
                    { "time": 0.0, "duration": 0.0, "value": { "label": "OUT", "comment": "Out 2" }, "confidence": None },
                    { "time": 0.0, "duration": 0.0, "value": { "label": "OUT", "comment": "Out 3" }, "confidence": None }
                ],
                "sandbox": {},
                "time": 0,
                "duration": duration
            }
        ],
        "file_metadata": {
            "title": track,
            "artist": artist,
            "release": str(row.get('Album', '')), # Use Album as release if available
            "duration": duration,
            "identifiers": {},
            "jams_version": "0.3.3"
        },
        "sandbox": {}
    }
    
    # Save to file
    try:
        with open(jams_path, 'w') as f:
            json.dump(blueprint, f, indent=2)
        return "Created", filename_base
    except Exception as e:
        return "Error", str(e)


if os.path.exists(csv_file_path):
    print("Generating Annotation Blueprints...")
    df = pd.read_csv(csv_file_path)
    
    created_count = 0
    exists_count = 0
    
    # Enable overwrite to update existing files with new template
    # Set to True since we just added the 3 IN / 3 OUT requirement
    force_update = True 
    
    for index, row in df.iterrows():
        status, name = create_jams_blueprint(row, overwrite=force_update)
        if status == "Created":
            created_count += 1
        elif status == "Exists":
            exists_count += 1
            
    print("-" * 40)
    print(f"Blueprints Created/Updated: {created_count}")
    print(f"Skipped (Existed):          {exists_count}")
    print(f"Total Processed:            {len(df)}")
    print("-" * 40)
    print(f"Saved to: {annotations_dir}")

else:
    print(f"CSV Failure: {csv_file_path} not found.")

Generating Annotation Blueprints...
----------------------------------------
Blueprints Created/Updated: 127
Skipped (Existed):          0
Total Processed:            127
----------------------------------------
Saved to: ../data/custom/annotations


In [ ]:
from annotator import AnnotatorTool
from IPython.display import display

print("Loading Annotation Tool...")
# Paths relative to the notebook location
app = AnnotatorTool(
    audio_dir="../data/custom/audio", 
    annotations_dir="../data/custom/annotations"
)
display(app.ui)

Loading Annotation Tool...
